In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 4 (Court Judgment Prediction): Architecture-1 decoder + layer freezing / gradual unfreezing
#
#   document -> its LAST 1536 tokens -> 3 chunks of 512 tokens
#            -> every chunk through the decoder
#            -> token vector = learned MIX of ALL layers (embeddings + every block)
#            -> one vector per chunk
#            -> BiLSTM over the chunks + attention pooling -> 0 / 1
#
# Fine-tuning technique: layer freezing / gradual unfreezing (ULMFiT, Howard & Ruder, 2018)
#   epoch 1     : every Phase-1 weight FROZEN, only the head trains (= the baseline)
#   epoch 2     : the top 1/4 of the blocks are unfrozen (their full weights train)
#   epoch 3     : the top 1/2
#   epoch 4 ... : all blocks (the token embeddings always stay frozen)
#   lower blocks get smaller learning rates (discriminative fine-tuning)
#
# Why the head reads ALL layers: on the real data, plain word counts (TF-IDF) on the same
# text reach dev AUC 0.71, the frozen LAST layer of Architecture-2 only 0.56 -> the last
# layer hides the words that decide a case; the lower layers keep them (probe: CELL 12).
#
# Trained on TRAIN only; dev picks the best epoch and the decision threshold.
# ============================================================

RUN_NAME = "p4_arch1_unfreeze"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 Architecture-1 checkpoint. If wrong, CELL 10 searches /kaggle/input for it.
CHECKPOINT_PATH = Path(
    "/kaggle/input/notebooks/iitianritik/decoder-only-archi-1/legal_lm/best_model.pt"
)

# Phase-4 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-4-cjpe/Phase_4_CJPE")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Which part of the (long) document the model reads
#   "end"       = the last DOC_TOKENS tokens (closest to the decision; default)
#   "start"     = the first DOC_TOKENS tokens (facts, background)
#   "start_end" = half from the start + half from the end
# ------------------------------------------------------------

DOC_PART = "end"
DOC_TOKENS = 1536            # tokens per document the model reads
CHUNK_LEN = 512              # every chunk goes through the decoder separately


# ------------------------------------------------------------
# Layer freezing / gradual unfreezing
#   lr of a block = LR_TOP_BLOCK * LAYER_LR_DECAY ** (number of blocks above it)
# ------------------------------------------------------------

UNFREEZE_FRACTIONS = [0.0, 0.25, 0.5, 1.0]   # per epoch: share of the blocks (from the top) that train;
                                             # the last value repeats. 0.0 = head only
LR_TOP_BLOCK = 3e-5          # learning rate of the top block (full weights, as in normal fine-tuning)
LAYER_LR_DECAY = 0.8         # every block gets 0.8 x the learning rate of the block above it
UNFREEZE_WARMUP = 200        # updates: a block that was just unfrozen warms its learning rate up from 0


# ------------------------------------------------------------
# Head:
#   token vectors -> mix of all layers (learned weights) -> STANDARDIZED (per feature, statistics of the Phase-1 model, CELL 12)
#   chunk vector  = [attention-pooled ; mean] of its token vectors, STANDARDIZED again
#                   (the attention LEARNS which tokens matter, e.g. "dismissed", "set aside")
#   case          = BiLSTM over the chunks -> attention pooling -> 2 classes
# ------------------------------------------------------------

POOL_SKIP_TOKENS = 4         # the first tokens of a chunk are not pooled (they behave abnormally in LMs)
LSTM_HIDDEN = 256
HEAD_DROPOUT = 0.1
LR_HEAD = 1e-3

CLASS_BALANCING = True       # loss weights = inverse class frequency (macro-F1 counts both classes)


# ------------------------------------------------------------
# Training (TRAIN data only)
# ------------------------------------------------------------

EPOCHS = 6                   # epoch = EPOCH_DOCS cases; epoch 1 = head only, then gradual unfreezing
EPOCH_DOCS = 8000             # cases per epoch, then dev is checked (all train cases are used before any repeats)
BATCH_DOCS = 4               # documents per forward pass (4 x 3 chunks x 512 tokens). If CUDA out-of-memory: 2
GRAD_ACCUM = 4               # -> 16 documents per weight update
PATIENCE = 3                 # stop if dev macro-F1 does not improve for 3 epochs (checked once every block is unfrozen)
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0

EVAL_CHUNKS = 48             # chunks per forward pass when predicting

# Before training (CELL 12): a linear probe on EVERY layer of the FROZEN Phase-1 model
# (PROBE_DOCS train cases, scored on dev), then token statistics from STATS_DOCS train cases (~4 minutes).
STATS_DOCS = 1000
PROBE_DOCS = 3000

# CELL 8: where is the signal? A plain word-count model (TF-IDF) on the start, the
# end and the whole case. Data analysis only (~7 minutes, does not use the model).
# Already measured on the real data: dev AUC start 0.612 | end 0.708 | whole 0.710;
# end macro-F1 0.653 (it does not depend on the model).
RUN_SIGNAL_CHECK = False
SIGNAL_TRAIN_DOCS = 15000


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 64              # per split

if DEBUG_MODE:
    EPOCHS = 2
    EPOCH_DOCS = 64
    PROBE_DOCS = 64
    STATS_DOCS = 64
    SIGNAL_TRAIN_DOCS = 64


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


assert DOC_PART in ("end", "start", "start_end")
assert DOC_TOKENS % CHUNK_LEN == 0

MAX_CHUNKS = DOC_TOKENS // CHUNK_LEN

print("Run       :", RUN_NAME)
print("Reads     :", f"{DOC_PART} of each document, {DOC_TOKENS} tokens = {MAX_CHUNKS} chunks x {CHUNK_LEN}")
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")

Run       : p4_arch1_unfreeze
Reads     : end of each document, 1536 tokens = 3 chunks x 512
Output    : /kaggle/working/p4_arch1_unfreeze
AMP dtype : torch.float16


In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER + FAST ENCODING
#
# Documents are long, so only the part we need is tokenized
# (e.g. the last ~DOC_TOKENS tokens), and every pre-token piece
# is BPE-encoded only once (cache).
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


piece_cache = {}      # pre-token piece -> token ids


def encode_fast(text):
    """Same ids as tokenizer.encode(text), but every piece is encoded only once."""

    ids = []

    for piece in tok_module.PRETOKEN_PATTERN.findall(tok_module.normalize_text(text)):

        if piece not in piece_cache:
            piece_cache[piece] = tokenizer.encode(piece)

        ids.extend(piece_cache[piece])

    return ids


def head_tokens(text, n):
    """First n tokens of text (tokenizes only the beginning, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[:n]

        part = text[:chars]
        part = part[:max(part.rfind(" "), 1)]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[:n]

        chars *= 2


def tail_tokens(text, n):
    """Last n tokens of text (tokenizes only the end, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[-n:]

        part = text[-chars:]
        part = part[max(part.find(" "), 0):]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[-n:]

        chars *= 2


demo = "The appeal is allowed. The order of the High Court dated 12.03.2010 is set aside."
assert encode_fast(demo) == tokenizer.encode(demo)
print("Demo:", [tokenizer.id_to_token[i] for i in encode_fast(demo)][:12])

Vocabulary size: 32004 | PAD / UNK: 0 1
Demo: ['The', ' ', 'appeal', ' ', 'is', ' ', 'allow', 'ed', '.', ' ', 'The', ' ']


In [4]:
# ============================================================
# CELL 4 — PHASE-4 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE4_FILES = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev":   "Phase_4_CJPE/Train/dev.json",
    "test":  "Phase_4_CJPE/Test/test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE4_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_4_CJPE/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE4_FILES.items():

        DATA_FILES[name] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )

for name, path in DATA_FILES.items():
    print(f"{name:6s} -> {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")

train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json  (737 MB)
dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json  (45 MB)
test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json  (35 MB)


In [5]:
# ============================================================
# CELL 5 — LOAD CASES + STATISTICS
#
# Every case: {"id", "text", "label" (0 / 1, or None for test)}
# ============================================================

LABEL_KEYS = ["label", "labels", "judgment", "decision", "y"]


def read_json(path):
    """A JSON file, or a JSON-lines file (one case per line)."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return [json.loads(line) for line in content.splitlines() if line.strip()]


def load_cases(path, with_labels=True):

    raw = read_json(path)

    # accepted layouts: [case, ...]  |  {"data": [case, ...]}  |  {case_id: case}
    if isinstance(raw, dict):

        list_key = next((k for k in ("data", "cases", "documents") if isinstance(raw.get(k), list)), None)

        if list_key is not None:
            raw = raw[list_key]
        else:
            raw = [dict(value, id=key) if isinstance(value, dict) else {"id": key, "text": value}
                   for key, value in raw.items()]

    cases = []

    for i, item in enumerate(raw):

        label_key = next((k for k in LABEL_KEYS if k in item), None) if with_labels else None
        label = item[label_key] if label_key is not None else None

        if isinstance(label, list):                 # e.g. [1]
            label = label[0]

        cases.append({
            "id": str(item.get("id", i)),
            "text": str(item.get("text", "")),
            "label": None if label is None else int(label),
        })

    if DEBUG_MODE:
        cases = cases[:DEBUG_DOCS]

    return cases


train_cases = load_cases(DATA_FILES["train"])
dev_cases = load_cases(DATA_FILES["dev"])
test_cases = load_cases(DATA_FILES["test"], with_labels=False)     # test labels are private: never read

for name, cases in [("train", train_cases), ("dev", dev_cases)]:
    assert all(c["label"] in (0, 1) for c in cases), f"{name}: labels must be 0 or 1"

ids = [c["id"] for c in test_cases]
assert len(ids) == len(set(ids)), "test ids are not unique"


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(f"{'split':6s} {'cases':>7s} {'label 0':>8s} {'label 1':>8s} {'chars (median / 90% / max)':>30s}")

for name, cases in [("train", train_cases), ("dev", dev_cases), ("test", test_cases)]:

    labels = Counter(c["label"] for c in cases)
    chars = np.array([len(c["text"]) for c in cases])

    print(f"{name:6s} {len(cases):7d} {labels.get(0, 0):8d} {labels.get(1, 0):8d} "
          f"{np.median(chars):12.0f} / {np.percentile(chars, 90):8.0f} / {chars.max():8d}")

empty = sum(not c["text"].strip() for c in train_cases + dev_cases + test_cases)
print("\nEmpty texts:", empty)

print("\nSample:", train_cases[0]["id"], "| label", train_cases[0]["label"])
print("  start:", repr(train_cases[0]["text"][:200]))
print("  end  :", repr(train_cases[0]["text"][-200:]))


# ------------------------------------------------------------
# Class weights for the loss (inverse frequency, mean 1)
# ------------------------------------------------------------

train_label_counts = Counter(c["label"] for c in train_cases)

if CLASS_BALANCING:
    CLASS_WEIGHTS = torch.tensor([len(train_cases) / (2 * max(train_label_counts[k], 1)) for k in (0, 1)])
else:
    CLASS_WEIGHTS = torch.ones(2)

print("\nClass weights:", [round(w, 3) for w in CLASS_WEIGHTS.tolist()])

split    cases  label 0  label 1     chars (median / 90% / max)
train    37387    22067    15320        13023 /    38996 /  2492701
dev       1988      994      994        16700 /    40495 /   260198
test      1517        0        0        17320 /    42859 /   233000

Empty texts: 0

Sample: 2020_1 | label 0

Class weights: [0.847, 1.22]


In [6]:
# ============================================================
# CELL 6 — TOKENIZE (only the part the model reads) + CHUNKS
#
# DOC_PART = "end": the last DOC_TOKENS tokens, cut into chunks of
# CHUNK_LEN FROM THE END, so the last chunk always ends where the
# document ends.
# ============================================================

def split_chunks(ids, from_end):

    if from_end:
        return [ids[max(0, i - CHUNK_LEN):i] for i in range(len(ids), 0, -CHUNK_LEN)][::-1]

    return [ids[i:i + CHUNK_LEN] for i in range(0, len(ids), CHUNK_LEN)]


def make_chunks(text):
    """List of token-id arrays (at most MAX_CHUNKS, each at most CHUNK_LEN)."""

    if DOC_PART == "end":
        chunks = split_chunks(tail_tokens(text, DOC_TOKENS), from_end=True)

    elif DOC_PART == "start":
        chunks = split_chunks(head_tokens(text, DOC_TOKENS), from_end=False)

    else:  # start_end
        half = DOC_TOKENS // 2
        start = head_tokens(text, half)
        end = tail_tokens(text, half)

        if len(start) < half or len(end) < half:          # short document: read all of it once
            chunks = split_chunks(head_tokens(text, DOC_TOKENS), from_end=False)
        else:
            chunks = split_chunks(start, from_end=False) + split_chunks(end, from_end=True)

    chunks = [c for c in chunks if len(c) > 0] or [[UNK_ID]]      # empty text -> 1 token

    return [np.array(c, dtype=np.int32) for c in chunks]


t0 = time.time()

for split, cases in [("train", train_cases), ("dev", dev_cases), ("test", test_cases)]:

    for k, case in enumerate(cases):

        case["chunks"] = make_chunks(case["text"])
        case["n_tokens"] = sum(len(c) for c in case["chunks"])

        if (k + 1) % 5000 == 0:
            print(f"  {split}: {k + 1}/{len(cases)} cases | {time.time() - t0:.0f}s", flush=True)

print(f"Tokenized in {time.time() - t0:.0f}s")


# check: fast encoding == tokenizer.encode on a few real cases
for case in train_cases[:5]:
    piece = case["text"][-3000:]
    assert encode_fast(piece) == tokenizer.encode(piece)


n_tokens = np.array([c["n_tokens"] for c in train_cases])
print(f"\nTrain cases shorter than {DOC_TOKENS} tokens: {(n_tokens < DOC_TOKENS).mean():.1%}")
print(f"Chunks per case: {Counter(len(c['chunks']) for c in train_cases)}")
print(f"Tokens per epoch: {EPOCH_DOCS * DOC_TOKENS / 1e6:.1f}M "
      f"({min(EPOCH_DOCS, len(train_cases))} cases)")

  train: 5000/37387 cases | 9s
  train: 10000/37387 cases | 15s
  train: 15000/37387 cases | 23s
  train: 20000/37387 cases | 31s
  train: 25000/37387 cases | 38s
  train: 30000/37387 cases | 44s
  train: 35000/37387 cases | 52s
Tokenized in 61s

Train cases shorter than 1536 tokens: 15.5%
Chunks per case: Counter({3: 33517, 2: 2117, 1: 1753})
Tokens per epoch: 12.3M (8000 cases)


In [7]:
# ============================================================
# CELL 7 — METRICS + DECISION THRESHOLD
# ============================================================

def binary_scores(labels, preds):
    """Macro P / R / F1 over the 2 classes (the competition metric is macro-F1)."""

    p, r, f, _ = precision_recall_fscore_support(labels, preds, labels=[0, 1], average="macro", zero_division=0)

    return {"precision": float(p), "recall": float(r), "f1": float(f)}


def best_threshold(labels, probs):
    """Threshold on P(label = 1) that gives the best macro-F1."""

    best_t, best_f1 = 0.5, -1.0

    # fixed grid + quantiles of the probabilities (works even when they are all close together)
    candidates = np.unique(np.concatenate([np.arange(0.05, 0.951, 0.01), np.quantile(probs, np.linspace(0.02, 0.98, 97))]))

    for t in candidates:

        f1 = f1_score(labels, (probs >= t).astype(int), labels=[0, 1], average="macro", zero_division=0)

        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)

    return best_t, best_f1


_labels = np.array([0, 0, 1, 1])
_probs = np.array([0.2, 0.6, 0.7, 0.9])
print("check:", binary_scores(_labels, (_probs >= 0.5).astype(int)), "| best threshold:", best_threshold(_labels, _probs))

In [8]:
# ============================================================
# CELL 8 — WHERE IS THE SIGNAL?  (data analysis only, NOT a submission)
#
# A plain word-count model (TF-IDF + logistic regression, trained on
# TRAIN, scored on dev) on 3 parts of every case:
#   start  = the first N characters
#   end    = the last N characters
#   whole  = the whole case (first 300,000 characters)
# N = the length of the text the model reads (DOC_PART, DOC_TOKENS).
# It uses no pretrained model; it only shows WHERE in a judgment the
# outcome is visible and what score plain word counts reach.
# ============================================================

signal_check = {}

if RUN_SIGNAL_CHECK:

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression

    t0 = time.time()

    signal_train = random.Random(SEED).sample(train_cases, min(SIGNAL_TRAIN_DOCS, len(train_cases)))

    def read_length(case):
        # characters of the text the model reads
        return len(tokenizer.decode([int(t) for chunk in case["chunks"] for t in chunk]))

    def parts(case):
        n = read_length(case)
        return {"start": case["text"][:n], "end": case["text"][-n:], "whole": case["text"][:300_000]}

    train_parts = [parts(c) for c in signal_train]
    dev_parts = [parts(c) for c in dev_cases]

    y_train = np.array([c["label"] for c in signal_train])
    y_dev = np.array([c["label"] for c in dev_cases])

    print(f"{'part':8s} {'dev AUC':>8s} {'macro-F1':>9s} {'threshold':>10s}")

    for part in ["start", "end", "whole"]:

        vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=200_000,
                                     sublinear_tf=True, dtype=np.float32)

        X_train = vectorizer.fit_transform([p[part] for p in train_parts])
        X_dev = vectorizer.transform([p[part] for p in dev_parts])

        clf = LogisticRegression(C=4.0, solver="liblinear", class_weight="balanced")
        clf.fit(X_train, y_train)
        p_dev = clf.predict_proba(X_dev)[:, 1]

        t, f1 = best_threshold(y_dev, p_dev)
        signal_check[part] = {"auc": float(roc_auc_score(y_dev, p_dev)), "f1": f1}

        print(f"{part:8s} {signal_check[part]['auc']:8.4f} {f1:9.4f} {t:10.2f}  ({time.time() - t0:.0f}s)", flush=True)

    del train_parts, dev_parts, X_train, X_dev

else:
    print("Signal check skipped (RUN_SIGNAL_CHECK = False)")

Signal check skipped (RUN_SIGNAL_CHECK = False)


In [9]:
# ============================================================
# CELL 9 — MODEL
#
#   every chunk -> Architecture-1 decoder (Phase-1 weights: frozen at the start, then
#                  unfrozen block by block from the top, CELL 13)
#               -> token vector = learned mix of ALL layers (embeddings + every block)
#               -> token vectors standardized
#               -> [attention-pooled ; mean] of its token vectors = chunk vector
#   chunks of a case -> LayerNorm -> BiLSTM over the chunks (in order)
#                    -> attention pooling (learns WHICH chunks matter)
#                    -> 2 classes
#
# Layer names match the Architecture-1 notebook, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-1 DECODER (causal, RoPE)
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * rms * self.weight).to(x.dtype)


def build_rope_cache(block_size, head_dim, theta):

    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(block_size).float(), inv_freq)

    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):

    x1, x2 = x.float().chunk(2, dim=-1)

    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).to(x.dtype)


class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]

        self.n_heads = config["n_heads"]
        self.head_dim = d_model // self.n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

    def forward(self, x, cos, sin):

        B, T, C = x.shape

        q, k, v = self.qkv(x).split(C, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(self.q_norm(q), cos, sin)
        k = apply_rope(self.k_norm(k), cos, sin)

        # causal, as in pretraining. Padding is at the END, so real tokens never see it.
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        return self.out_proj(y.transpose(1, 2).contiguous().view(B, T, C))


class SwiGLU(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.gate = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.up = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.down = nn.Linear(config["d_ff"], config["d_model"], bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class TransformerBlock(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.norm1 = RMSNorm(config["d_model"])
        self.attn = CausalSelfAttention(config)
        self.norm2 = RMSNorm(config["d_model"])
        self.mlp = SwiGLU(config)

    def forward(self, x, cos, sin):

        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.mlp(self.norm2(x))

        return x


class DecoderBackbone(nn.Module):
    """Architecture-1 without the LM head: tokens -> hidden vectors."""

    def __init__(self, config):
        super().__init__()

        self.token_embedding = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config["n_layers"])])
        self.norm_f = RMSNorm(config["d_model"])

        cos, sin = build_rope_cache(config["block_size"], config["d_model"] // config["n_heads"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, input_ids, mask=None, all_layers=False):
        """`mask` is not needed: attention is causal and padding is at the end.
        all_layers=False: final hidden vectors (as in Phase 1).
        all_layers=True : list of the embeddings + the output of every block (before any norm)."""

        T = input_ids.size(1)

        cos, sin = self.rope_cos[:T], self.rope_sin[:T]

        x = self.token_embedding(input_ids)
        states = [x]

        for block in self.blocks:
            x = block(x, cos, sin)
            states.append(x)

        if all_layers:
            return states

        return self.norm_f(x)


# ============================================================
# Layer freezing / unfreezing
# ============================================================

def backbone_blocks(backbone):
    """The transformer blocks of the Phase-1 model, bottom -> top."""

    return list(backbone.blocks)


def set_trainable_blocks(model, n_top):
    """Freeze every Phase-1 weight, then unfreeze (full weights) the top n_top blocks."""

    for p in model.backbone.parameters():
        p.requires_grad = False

    blocks = backbone_blocks(model.backbone)

    for block in blocks[len(blocks) - n_top:]:
        for p in block.parameters():
            p.requires_grad = True

    model.train_backbone = n_top > 0          # no unfrozen block -> the Phase-1 model runs without gradients


def unfrozen_blocks(epoch):
    """Number of blocks (counted from the top) that train in this epoch (epochs count from 1)."""

    fraction = UNFREEZE_FRACTIONS[min(epoch, len(UNFREEZE_FRACTIONS)) - 1]

    return math.ceil(fraction * N_BLOCKS)


# ============================================================
# Standardization with RUNNING statistics (no learned weights)
#   x -> (x - mean) / std
# The statistics start from the frozen Phase-1 model (CELL 12) and then
# slowly follow the features while the unfrozen blocks change them (fixed statistics
# would make the features drift away from what the head has learned).
# ============================================================

class RunningStandardize(nn.Module):

    def __init__(self, dim, momentum=0.01):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("mean", torch.zeros(dim))
        self.register_buffer("std", torch.ones(dim))

    def forward(self, x, mask):
        """x [..., dim], mask [...] (True = real value)."""

        if self.training:
            with torch.no_grad():
                values = x[mask]
                if len(values) > 1:
                    self.mean.mul_(1 - self.momentum).add_(self.momentum * values.mean(dim=0))
                    self.std.mul_(1 - self.momentum).add_(self.momentum * values.std(dim=0).clamp(min=1e-4))

        return (x - self.mean) / self.std


# ============================================================
# Case classifier (hierarchical: tokens -> chunks -> case)
# ============================================================

class CaseClassifier(nn.Module):

    def __init__(self, backbone, d_model, n_states):
        super().__init__()

        self.backbone = backbone
        self.train_backbone = True          # False = the Phase-1 model runs without gradients (head-only epoch)

        # Layer mix: token vector = sum_l w_l * LayerNorm(h_l) over the embeddings and every
        # block, w = softmax(layer_logits). The last layer alone hides the words that decide a
        # case (Architecture-2: frozen probe AUC 0.56 vs 0.71 for word counts); the lower layers
        # keep them. Start values: CELL 12 (per-layer probe); then learned with the head.
        self.layer_logits = nn.Parameter(torch.zeros(n_states))

        # Token vectors of a language model share one big direction (chunks of
        # different cases have cosine ~0.97); what tells cases apart is small.
        # Standardizing every feature (minus its mean over train tokens, divided
        # by its std) keeps only that part. Statistics: CELL 12.
        self.token_standardize = RunningStandardize(d_model)

        # token attention: LEARNS which tokens of a chunk matter (an average over
        # 512 tokens drowns the few words that decide a case)
        self.token_attention = nn.Sequential(nn.Linear(d_model, 128), nn.Tanh(), nn.Linear(128, 1))

        # chunk vectors are standardized again (statistics over train chunks, CELL 12):
        # this is what lets the head see the small differences between cases
        self.chunk_standardize = RunningStandardize(2 * d_model)

        self.vector_norm = nn.LayerNorm(2 * d_model)                  # [attention-pooled ; mean] -> 2 * d_model
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        self.lstm = nn.LSTM(2 * d_model, LSTM_HIDDEN, batch_first=True, bidirectional=True)

        self.attention = nn.Sequential(nn.Linear(2 * LSTM_HIDDEN, 128), nn.Tanh(), nn.Linear(128, 1))
        self.classifier = nn.Linear(2 * LSTM_HIDDEN, 2)

        self.register_buffer("class_weights", CLASS_WEIGHTS.clone().float())

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def layer_weights(self):
        return torch.softmax(self.layer_logits.float(), dim=0)

    def token_states(self, ids, mask):
        """[n_chunks, T] ids -> [n_chunks, T, D] float32: mix of the LayerNorm-ed hidden states of all layers."""

        with torch.set_grad_enabled(torch.is_grad_enabled() and self.train_backbone):
            with autocast():
                states = self.backbone(ids, mask, all_layers=True)

        mixed = 0.0
        for w, h in zip(self.layer_weights(), states):
            mixed = mixed + w * F.layer_norm(h.float(), h.shape[-1:])

        return mixed

    def chunk_vectors(self, chunks):
        """List of token-id arrays -> [n_chunks, 2D] float32.
        Chunk vector = [attention-pooled ; mean] of its standardized token vectors."""

        max_len = max(len(c) for c in chunks)
        ids = torch.full((len(chunks), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(chunks), max_len), dtype=torch.bool)

        for row, c in enumerate(chunks):
            ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
            mask[row, :len(c)] = True

        ids, mask = ids.to(DEVICE), mask.to(DEVICE)

        hidden = self.token_states(ids, mask)                         # [n_chunks, T, D], mix of all layers

        # do not pool the first POOL_SKIP_TOKENS tokens (unless the chunk is very short)
        pool_mask = mask.clone()
        long_chunk = mask.sum(dim=1) > 2 * POOL_SKIP_TOKENS
        pool_mask[long_chunk, :POOL_SKIP_TOKENS] = False

        z = self.token_standardize(hidden, pool_mask)                  # standardized token vectors

        scores = self.token_attention(z).squeeze(-1).masked_fill(~pool_mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)                         # [n_chunks, T]
        attended = (weights.unsqueeze(-1) * z).sum(dim=1)

        m = pool_mask.unsqueeze(-1)
        mean = (z * m).sum(dim=1) / m.sum(dim=1)

        return torch.cat([attended, mean], dim=-1)

    def classify(self, case_vectors):
        """
        case_vectors: list of [n_chunks_i, 2D] tensors (one per case, chunks in document order)
        returns logits [n_cases, 2] and attention weights [n_cases, max_chunks]
        """

        n_chunks = [len(v) for v in case_vectors]
        K = max(n_chunks)

        x = torch.zeros(len(case_vectors), K, case_vectors[0].size(-1), device=DEVICE)
        mask = torch.zeros(len(case_vectors), K, dtype=torch.bool, device=DEVICE)

        for i, v in enumerate(case_vectors):
            x[i, :len(v)] = v.to(DEVICE)
            mask[i, :len(v)] = True

        x = self.chunk_standardize(x, mask)
        x = self.dropout(self.vector_norm(x))

        lengths = torch.tensor(n_chunks)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=K)    # [n, K, 2H]

        scores = self.attention(out).squeeze(-1).masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)                                            # [n, K]

        case_vector = (weights.unsqueeze(-1) * out).sum(dim=1)

        return self.classifier(self.dropout(case_vector)), weights

    def forward(self, cases):

        all_chunks = [c for case in cases for c in case["chunks"]]
        vectors = self.chunk_vectors(all_chunks)

        case_vectors, k = [], 0
        for case in cases:
            n = len(case["chunks"])
            case_vectors.append(vectors[k:k + n])
            k += n

        return self.classify(case_vectors)

    def loss(self, cases):

        logits, _ = self(cases)
        labels = torch.tensor([case["label"] for case in cases], device=DEVICE)

        return F.cross_entropy(logits, labels, weight=self.class_weights)


In [10]:
# ============================================================
# CELL 10 — BUILD THE MODEL FROM THE PHASE-1 ARCHITECTURE-1 CHECKPOINT
# ============================================================

def is_architecture1(path):

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        return isinstance(ckpt, dict) and "rope_theta" in ckpt.get("model_config", {})
    except Exception:
        return False


if not CHECKPOINT_PATH.exists():

    found = [p for p in sorted(Path("/kaggle/input").rglob("best_model.pt")) if is_architecture1(p)]
    assert found, f"{CHECKPOINT_PATH} not found and no Architecture-1 checkpoint under /kaggle/input"

    CHECKPOINT_PATH = found[0]
    print("Checkpoint found at:", CHECKPOINT_PATH)


phase1 = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
MODEL_CONFIG = dict(phase1["model_config"])

# weights without the LM head (tied to the embedding, not needed here)
MODEL_WEIGHTS = {
    key.replace("_orig_mod.", ""): value
    for key, value in phase1["model_state_dict"].items()
    if not key.replace("_orig_mod.", "").startswith("lm_head.")
}

del phase1

D_MODEL = MODEL_CONFIG["d_model"]

print("Phase-1 config:", MODEL_CONFIG)

assert MODEL_CONFIG["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert CHUNK_LEN <= MODEL_CONFIG["block_size"], f"CHUNK_LEN must be <= {MODEL_CONFIG['block_size']}"


def build_model():
    """Phase-1 decoder (every weight frozen at the start) + new case classifier."""

    backbone = DecoderBackbone(MODEL_CONFIG)
    backbone.load_state_dict(MODEL_WEIGHTS, strict=True)          # every name and shape must match

    for p in backbone.parameters():
        p.requires_grad = False

    # the hidden states of all layers; the last one (after the final norm) = the Phase-1 output
    test_ids = torch.randint(4, VOCAB_SIZE, (2, 24))
    test_mask = torch.ones(2, 24, dtype=torch.bool)
    test_mask[1, 16:] = False                                    # second row: padding at the end

    backbone.eval()
    with torch.no_grad():
        states = backbone(test_ids, test_mask, all_layers=True)
        assert len(states) == len(backbone_blocks(backbone)) + 1
        assert torch.allclose(backbone.norm_f(states[-1]), backbone(test_ids, test_mask), atol=1e-5)
    backbone.train()

    return CaseClassifier(backbone, D_MODEL, len(backbone_blocks(backbone)) + 1).to(DEVICE)


# free GPU memory if this cell is re-run
for name in ["model", "optimizer", "scaler"]:
    globals().pop(name, None)

sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

model = build_model()

N_BLOCKS = len(backbone_blocks(model.backbone))

# check: unfreezing the top 2 blocks makes exactly their weights trainable, nothing else in the Phase-1 model
set_trainable_blocks(model, 2)
_trainable = {name for name, p in model.backbone.named_parameters() if p.requires_grad}
_expected = {f"blocks.{i}.{name}" for i in (N_BLOCKS - 2, N_BLOCKS - 1)
             for name, _ in backbone_blocks(model.backbone)[i].named_parameters()}
assert _trainable == _expected, "unfreezing selected the wrong weights"

set_trainable_blocks(model, 0)
assert not any(p.requires_grad for p in model.backbone.parameters())

n_total = sum(p.numel() for p in model.parameters())
n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_head = sum(p.numel() for p in model.head_parameters())
block_sizes = [sum(p.numel() for p in block.parameters()) for block in backbone_blocks(model.backbone)]

print(f"\nPhase-1 model            : {n_backbone:,}  ({N_BLOCKS} blocks of {block_sizes[0]:,})")
print(f"Head (trainable)         : {n_head:,}")

print("\nUnfreezing plan:")
for epoch in range(1, EPOCHS + 1):
    n_top = unfrozen_blocks(epoch)
    if n_top == 0:
        print(f"  epoch {epoch}: head only (every Phase-1 weight frozen)")
    else:
        lowest_lr = LR_TOP_BLOCK * LAYER_LR_DECAY ** (n_top - 1)
        print(f"  epoch {epoch}: top {n_top}/{N_BLOCKS} blocks train ({sum(block_sizes[N_BLOCKS - n_top:]):,} weights, "
              f"lr {LR_TOP_BLOCK:.1e} (top) ... {lowest_lr:.1e})")


Phase-1 config: {'vocab_size': 32004, 'block_size': 2048, 'd_model': 768, 'n_layers': 12, 'n_heads': 12, 'd_ff': 2048, 'rope_theta': 10000.0}

Phase-1 model            : 109,534,464  (12 blocks of 7,079,552)
Head (trainable)         : 3,842,577

Unfreezing plan:
  epoch 1: head only (every Phase-1 weight frozen)
  epoch 2: top 3/12 blocks train (21,238,656 weights, lr 3.0e-05 (top) ... 1.9e-05)
  epoch 3: top 6/12 blocks train (42,477,312 weights, lr 3.0e-05 (top) ... 9.8e-06)
  epoch 4: top 12/12 blocks train (84,954,624 weights, lr 3.0e-05 (top) ... 2.6e-06)
  epoch 5: top 12/12 blocks train (84,954,624 weights, lr 3.0e-05 (top) ... 2.6e-06)
  epoch 6: top 12/12 blocks train (84,954,624 weights, lr 3.0e-05 (top) ... 2.6e-06)


In [11]:
# ============================================================
# CELL 11 — PREDICTION + EVALUATION
#
# Chunk vectors are computed once per case (in big batches), then
# the classifier turns them into P(label = 1).
# ============================================================

@torch.no_grad()
def case_vectors(model, cases):
    """List of [n_chunks, 2D] CPU tensors, one per case."""

    model.eval()

    all_chunks = [c for case in cases for c in case["chunks"]]
    vectors = []

    for k in range(0, len(all_chunks), EVAL_CHUNKS):
        vectors.append(model.chunk_vectors(all_chunks[k:k + EVAL_CHUNKS]).cpu())

    vectors = torch.cat(vectors)

    out, k = [], 0
    for case in cases:
        n = len(case["chunks"])
        out.append(vectors[k:k + n])
        k += n

    model.train()

    return out


@torch.no_grad()
def classify_vectors(model, vectors, batch=256):
    """Chunk vectors -> (P(label = 1) [n_cases], attention weights list)."""

    model.eval()

    probs, weights = [], []

    for k in range(0, len(vectors), batch):

        logits, w = model.classify(vectors[k:k + batch])

        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu())
        weights.extend([w[i, :len(v)].cpu().numpy() for i, v in enumerate(vectors[k:k + batch])])

    model.train()

    return torch.cat(probs).numpy(), weights


def evaluate(model, cases):
    """Dev scores at threshold 0.5 and at the best threshold."""

    labels = np.array([c["label"] for c in cases])
    probs, _ = classify_vectors(model, case_vectors(model, cases))

    threshold, _ = best_threshold(labels, probs)

    return {
        "at_0.5": binary_scores(labels, (probs >= 0.5).astype(int)),
        "threshold": threshold,
        "at_threshold": binary_scores(labels, (probs >= threshold).astype(int)),
        "auc": float(roc_auc_score(labels, probs)) if len(set(labels)) == 2 else 0.0,
    }, probs

In [12]:
# ============================================================
# CELL 12 — PHASE-1 FEATURES: WHICH LAYER KNOWS THE OUTCOME? + TOKEN STATISTICS (before training)
#
#   1. linear probe on EVERY layer of the FROZEN Phase-1 model (embeddings, layer 1, layer 2, ...):
#      case vector = mean of the LayerNorm-ed token vectors of that layer
#      -> logistic regression on PROBE_DOCS train cases -> dev AUC / macro-F1.
#      The layer mix of the classifier STARTS from these scores
#      (better layer -> larger weight); training then adjusts it.
#   2. token statistics of the MIXED token vectors (STATS_DOCS train cases)
#      -> stored in the model; then the same for the chunk vectors
#   3. how similar are chunks of different cases? (raw / standardized)
# ============================================================

from sklearn.linear_model import LogisticRegression

t0 = time.time()

N_STATES = len(model.layer_logits)
LAYER_NAMES = ["embeddings"] + [f"layer {i}" for i in range(1, N_STATES)]


def chunk_batches(chunks):
    """Padded batches of EVAL_CHUNKS chunks: (ids, attention mask, pooling mask without the first tokens)."""

    for k in range(0, len(chunks), EVAL_CHUNKS):

        batch = chunks[k:k + EVAL_CHUNKS]
        max_len = max(len(c) for c in batch)
        ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(batch), max_len), dtype=torch.bool)

        for row, c in enumerate(batch):
            ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
            mask[row, :len(c)] = True

        pool_mask = mask.clone()
        long_chunk = mask.sum(dim=1) > 2 * POOL_SKIP_TOKENS
        pool_mask[long_chunk, :POOL_SKIP_TOKENS] = False

        yield ids.to(DEVICE), mask.to(DEVICE), pool_mask.to(DEVICE)


@torch.no_grad()
def layer_means(cases):
    """[n_cases, N_STATES, D]: per case and layer, the mean LayerNorm-ed token vector (averaged over its chunks)."""

    model.eval()

    chunks = [c for case in cases for c in case["chunks"]]
    per_chunk = []

    for ids, mask, pool_mask in chunk_batches(chunks):

        with autocast():
            states = model.backbone(ids, mask, all_layers=True)

        m = pool_mask.unsqueeze(-1)
        per_chunk.append(torch.stack(
            [(F.layer_norm(h.float(), h.shape[-1:]) * m).sum(dim=1) / m.sum(dim=1) for h in states], dim=1).cpu())

    model.train()

    per_chunk = torch.cat(per_chunk)

    out, k = [], 0
    for case in cases:
        n = len(case["chunks"])
        out.append(per_chunk[k:k + n].mean(dim=0))
        k += n

    return torch.stack(out)


# ---------------- 1. probe every layer ----------------
probe_cases = random.Random(SEED + 1).sample(train_cases, min(PROBE_DOCS, len(train_cases)))

L_probe = layer_means(probe_cases)
L_dev = layer_means(dev_cases)
y_probe = np.array([c["label"] for c in probe_cases])
y_dev = np.array([c["label"] for c in dev_cases])

print(f"Linear probe on the frozen Phase-1 model ({len(probe_cases)} train cases -> dev):")
print(f"  {'layer':12s} {'dev AUC':>8s} {'macro-F1':>9s}")

probe_results = {}

for l, name in enumerate(LAYER_NAMES):

    mu = L_probe[:, l].mean(dim=0)
    sd = L_probe[:, l].std(dim=0).clamp(min=1e-6)
    X_train = ((L_probe[:, l] - mu) / sd).numpy()
    X_dev = ((L_dev[:, l] - mu) / sd).numpy()

    best = None
    for C in (0.01, 0.1):
        clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
        clf.fit(X_train, y_probe)
        p_dev = clf.predict_proba(X_dev)[:, 1]
        auc_l = float(roc_auc_score(y_dev, p_dev))
        if best is None or auc_l > best["auc"]:
            t, f1 = best_threshold(y_dev, p_dev)
            best = {"auc": auc_l, "f1": f1, "threshold": t, "C": C}

    probe_results[name] = best
    print(f"  {name:12s} {best['auc']:8.4f} {best['f1']:9.4f}")

PROBE_BEST_LAYER = max(probe_results, key=lambda n: probe_results[n]["auc"])
PROBE_BEST = probe_results[PROBE_BEST_LAYER]

# the layer mix starts from the probe: weight ~ exp(20 * (AUC - 0.5))
start_logits = torch.tensor([20.0 * max(probe_results[n]["auc"] - 0.5, 0.0) for n in LAYER_NAMES])

with torch.no_grad():
    model.layer_logits.copy_(start_logits.to(DEVICE))

print(f"\nBest frozen layer: {PROBE_BEST_LAYER} (dev AUC {PROBE_BEST['auc']:.4f}, macro-F1 {PROBE_BEST['f1']:.4f})")
print("Layer mix at the start:", {n: round(float(w), 3) for n, w in zip(LAYER_NAMES, model.layer_weights().detach())})
print(f"({time.time() - t0:.0f}s)")


# ---------------- 2. token statistics of the mixed token vectors ----------------
stat_cases = random.Random(SEED).sample(train_cases, min(STATS_DOCS, len(train_cases)))
stat_chunks = [c for case in stat_cases for c in case["chunks"]]

total = torch.zeros(D_MODEL, dtype=torch.float64, device=DEVICE)
total_sq = torch.zeros_like(total)
count = 0
raw_chunk_means = []

model.eval()

with torch.no_grad():

    for ids, mask, pool_mask in chunk_batches(stat_chunks):

        hidden = model.token_states(ids, mask)

        m = pool_mask.unsqueeze(-1)
        total += (hidden * m).sum(dim=(0, 1)).double()
        total_sq += (hidden.pow(2) * m).sum(dim=(0, 1)).double()
        count += int(pool_mask.sum())

        raw_chunk_means.append(((hidden * m).sum(dim=1) / m.sum(dim=1)).cpu())

model.train()

TOKEN_MEAN = (total / count).float()
TOKEN_STD = (total_sq / count - (total / count).pow(2)).clamp(min=1e-8).sqrt().float().clamp(min=1e-4)

model.token_standardize.mean.copy_(TOKEN_MEAN)
model.token_standardize.std.copy_(TOKEN_STD)

print(f"\nToken statistics from {count:,} tokens of {len(stat_cases)} train cases")

# chunk statistics (same cases, now with standardized tokens)
stat_vectors = torch.cat(case_vectors(model, stat_cases))

FEATURE_MEAN = stat_vectors.mean(dim=0)
FEATURE_STD = stat_vectors.std(dim=0).clamp(min=1e-4)

model.chunk_standardize.mean.copy_(FEATURE_MEAN.to(DEVICE))
model.chunk_standardize.std.copy_(FEATURE_STD.to(DEVICE))


# ---------------- 3. how similar are chunks of DIFFERENT cases? ----------------
raw = torch.cat(raw_chunk_means)                                   # mean of the mixed token vectors
std = (stat_vectors - FEATURE_MEAN) / FEATURE_STD                  # what the head sees

_i = torch.randint(0, len(raw), (1000,))
_j = torch.randint(0, len(raw), (1000,))
cos_raw = F.cosine_similarity(raw[_i], raw[_j], dim=-1).mean().item()
cos_std = F.cosine_similarity(std[_i], std[_j], dim=-1).mean().item()

print(f"mean cosine between random chunks: raw {cos_raw:.3f} -> standardized {cos_std:.3f}")
print(f"CELL 12 done in {time.time() - t0:.0f}s")

del raw, std, raw_chunk_means, stat_vectors, L_probe, L_dev


Linear probe on the frozen Phase-1 model (3000 train cases -> dev):
  layer         dev AUC  macro-F1
  embeddings     0.6014    0.5788
  layer 1        0.5794    0.5609
  layer 2        0.5827    0.5694
  layer 3        0.5643    0.5442
  layer 4        0.5785    0.5572
  layer 5        0.5886    0.5719
  layer 6        0.5918    0.5835
  layer 7        0.5780    0.5565
  layer 8        0.5877    0.5648
  layer 9        0.5925    0.5716
  layer 10       0.5876    0.5722
  layer 11       0.5884    0.5673
  layer 12       0.5905    0.5815

Best frozen layer: embeddings (dev AUC 0.6014, macro-F1 0.5788)
(192s)

Token statistics from 1,423,202 tokens of 1000 train cases
mean cosine between random chunks: raw 0.778 -> standardized 0.001
CELL 12 done in 259s


In [13]:
# ============================================================
# CELL 13 — OPTIMIZER + TRAINING FUNCTION (gradual unfreezing)
#   one parameter group per Phase-1 block (bottom -> top), each with its own
#   learning rate (lower blocks learn more slowly), + one group for the head.
#   Epoch e: the top unfrozen_blocks(e) blocks train; every other Phase-1 weight is frozen.
# ============================================================

def make_optimizer(model):
    """Groups 0 .. N_BLOCKS-1 = Phase-1 blocks (bottom -> top), last group = head."""

    groups = []

    for i, block in enumerate(backbone_blocks(model.backbone)):
        groups.append({"params": list(block.parameters()),
                       "lr": LR_TOP_BLOCK * LAYER_LR_DECAY ** (N_BLOCKS - 1 - i), "weight_decay": 0.01})

    groups.append({"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.01})

    return torch.optim.AdamW(groups, betas=(0.9, 0.999))


def set_lr(optimizer, base_lrs, step, total_steps, unfrozen_at):
    """Linear warmup, then linear decay to 0. A block that was just unfrozen warms
    its rate up again over UNFREEZE_WARMUP updates; frozen blocks get rate 0."""

    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))
    decay = 1.0 if step < warmup_steps else max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

    optimizer.param_groups[-1]["lr"] = base_lrs[-1] * min(1.0, (step + 1) / warmup_steps) * decay

    for i in range(N_BLOCKS):
        if unfrozen_at[i] is None:
            optimizer.param_groups[i]["lr"] = 0.0
        else:
            warm = min(1.0, (step + 1 - unfrozen_at[i]) / UNFREEZE_WARMUP)
            optimizer.param_groups[i]["lr"] = base_lrs[i] * warm * decay


def trainable_state(model):
    """The weights that train now (head + unfrozen blocks) + the running feature statistics."""

    state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
    state.update({name: b.detach().cpu().clone() for name, b in model.named_buffers() if "standardize" in name})

    return state


def restore(model, state):
    """Back to a saved state: first every Phase-1 weight as in the checkpoint,
    then the weights that had been trained up to that epoch."""

    model.backbone.load_state_dict(MODEL_WEIGHTS, strict=True)
    model.load_state_dict(state, strict=False)


def train_model(model, epochs):
    """
    Trains on TRAIN cases only. Every epoch uses EPOCH_DOCS cases (the train set is
    shuffled once and walked through, so every case is used before any repeats).
    Gradual unfreezing: epoch e trains the head + the top unfrozen_blocks(e) blocks.
    After every epoch: dev macro-F1 (at the best threshold); the best epoch is kept.
    """

    docs_per_epoch = min(EPOCH_DOCS, len(train_cases))
    updates_per_epoch = math.ceil(docs_per_epoch / (BATCH_DOCS * GRAD_ACCUM))
    total_steps = epochs * updates_per_epoch

    optimizer = make_optimizer(model)
    base_lrs = [group["lr"] for group in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    unfrozen_at = [None] * N_BLOCKS          # update step at which every block was unfrozen

    rng = random.Random(SEED)
    order = list(range(len(train_cases)))
    rng.shuffle(order)
    cursor = 0

    history = []
    best_score, best_epoch, best_state, best_top = -1.0, 0, None, 0
    step = 0
    t_start = time.time()

    model.train()

    for epoch in range(1, epochs + 1):

        # the next docs_per_epoch cases of the shuffled order
        if cursor + docs_per_epoch > len(order):
            rng.shuffle(order)
            cursor = 0

        epoch_cases = [train_cases[i] for i in order[cursor:cursor + docs_per_epoch]]

        # gradual unfreezing: the top n_top blocks train in this epoch
        n_top = unfrozen_blocks(epoch)
        set_trainable_blocks(model, n_top)

        for i in range(N_BLOCKS - n_top, N_BLOCKS):
            if unfrozen_at[i] is None:
                unfrozen_at[i] = step
        cursor += docs_per_epoch

        batches = [epoch_cases[k:k + BATCH_DOCS] for k in range(0, len(epoch_cases), BATCH_DOCS)]

        total_loss, skipped = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for b, batch in enumerate(batches):

            loss = model.loss(batch)
            scaler.scale(loss / GRAD_ACCUM).backward()
            total_loss += loss.item()

            if (b + 1) % GRAD_ACCUM == 0 or b + 1 == len(batches):

                set_lr(optimizer, base_lrs, step, total_steps, unfrozen_at)

                scaler.unscale_(optimizer)

                # clip the Phase-1 blocks and the head separately
                block_params = [p for group in optimizer.param_groups[:-1] for p in group["params"] if p.grad is not None]
                if block_params:
                    torch.nn.utils.clip_grad_norm_(block_params, GRAD_CLIP)
                torch.nn.utils.clip_grad_norm_(optimizer.param_groups[-1]["params"], GRAD_CLIP)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped += scaler.get_scale() < scale_before

                optimizer.zero_grad(set_to_none=True)
                step += 1

            if (b + 1) % 500 == 0:
                print(f"    batch {b + 1}/{len(batches)} | loss {total_loss / (b + 1):.4f} | "
                      f"{(time.time() - t_start) / 60:.1f} min", flush=True)

        dev_scores, _ = evaluate(model, dev_cases)

        log = {"epoch": epoch, "loss": total_loss / len(batches), "dev": dev_scores, "skipped_steps": int(skipped)}

        stage = "head only" if n_top == 0 else f"top {n_top}/{N_BLOCKS} blocks"
        log["blocks_trained"] = n_top

        line = (f"Epoch {epoch}/{epochs} ({stage}) | loss {log['loss']:.4f} | "
                f"dev macro-F1 @0.5 {dev_scores['at_0.5']['f1']:.4f} | "
                f"@{dev_scores['threshold']:.2f} {dev_scores['at_threshold']['f1']:.4f} | AUC {dev_scores['auc']:.4f}")

        stop = False

        if dev_scores["at_threshold"]["f1"] > best_score:
            best_score, best_epoch, best_state = dev_scores["at_threshold"]["f1"], epoch, trainable_state(model)
            best_top = n_top
            line += "  <- best"
        elif epoch - best_epoch >= PATIENCE and epoch >= len(UNFREEZE_FRACTIONS):
            stop = True

        if skipped:
            line += f" | skipped {skipped}"
        line += f" | {(time.time() - t_start) / 60:.1f} min"

        print(line, flush=True)
        history.append(log)

        mix = " ".join(f"{w:.2f}" for w in model.layer_weights().detach().tolist())
        print(f"    layer mix (embeddings, layer 1..): {mix}")

        if epoch == 1 and dev_scores["auc"] < PROBE_BEST["auc"] - 0.03:
            print(f"WARNING: frozen Phase-1 model + head reaches dev AUC {dev_scores['auc']:.4f}, below the best "
                  f"layer probe ({PROBE_BEST['auc']:.4f}) -> send me CELL 12 and this log.")

        if stop:
            print(f"Early stop: no improvement for {PATIENCE} epochs.")
            break

    restore(model, best_state)                    # back to the best epoch
    set_trainable_blocks(model, best_top)         # trainable_state() = the weights trained up to the best epoch
    model.train_backbone = True

    return history, best_epoch


In [14]:
# ============================================================
# CELL 14 — TRAIN (TRAIN data only; dev picks the best epoch)
# ============================================================

print(f"{RUN_NAME} | train {len(train_cases)} cases ({min(EPOCH_DOCS, len(train_cases))} per epoch) "
      f"-> dev {len(dev_cases)} cases")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
history, best_epoch = train_model(model, EPOCHS)
train_minutes = (time.time() - t0) / 60

peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(model), OUTPUT_DIR / "best_trainable.pt")

print(f"\n✅ Trained in {train_minutes:.1f} min | best epoch {best_epoch} | peak GPU {peak_gpu_gb:.2f} GB")

p4_arch1_unfreeze | train 37387 cases (8000 per epoch) -> dev 1988 cases
    batch 500/2000 | loss 0.6965 | 1.2 min
    batch 1000/2000 | loss 0.6842 | 2.4 min
    batch 1500/2000 | loss 0.6630 | 3.6 min
    batch 2000/2000 | loss 0.6514 | 4.7 min
Epoch 1/6 (head only) | loss 0.6514 | dev macro-F1 @0.5 0.5085 | @0.36 0.6035 | AUC 0.6425  <- best | 5.9 min
    layer mix (embeddings, layer 1..): 0.09 0.06 0.07 0.05 0.07 0.08 0.09 0.07 0.08 0.09 0.08 0.08 0.08
    batch 500/2000 | loss 0.6162 | 7.6 min
    batch 1000/2000 | loss 0.6114 | 9.3 min
    batch 1500/2000 | loss 0.6058 | 11.1 min
    batch 2000/2000 | loss 0.6041 | 12.8 min
Epoch 2/6 (top 3/12 blocks) | loss 0.6041 | dev macro-F1 @0.5 0.5597 | @0.39 0.6467 | AUC 0.6824  <- best | 14.0 min
    layer mix (embeddings, layer 1..): 0.09 0.06 0.07 0.05 0.07 0.08 0.09 0.07 0.09 0.09 0.08 0.08 0.07
    batch 500/2000 | loss 0.5978 | 16.3 min
    batch 1000/2000 | loss 0.5965 | 18.6 min
    batch 1500/2000 | loss 0.5908 | 20.8 min
    ba

In [15]:
# ============================================================
# CELL 15 — DEV REPORT
#   scores + threshold, confusion matrix, errors by document length,
#   which chunks the model uses, false positives / negatives
# ============================================================

t0 = time.time()
dev_vectors = case_vectors(model, dev_cases)
dev_probs, dev_attention = classify_vectors(model, dev_vectors)
predict_seconds = time.time() - t0

dev_labels = np.array([c["label"] for c in dev_cases])

THRESHOLD, _ = best_threshold(dev_labels, dev_probs)        # used for the test predictions (CELL 16)
dev_preds = (dev_probs >= THRESHOLD).astype(int)

at_05 = binary_scores(dev_labels, (dev_probs >= 0.5).astype(int))
at_t = binary_scores(dev_labels, dev_preds)
auc = float(roc_auc_score(dev_labels, dev_probs))

print(f"DEV ({RUN_NAME}, best epoch {best_epoch})")
print(f"{'':16s} {'macro-P':>8s} {'macro-R':>8s} {'macro-F1':>9s}")
print(f"{'threshold 0.50':16s} {at_05['precision']:8.4f} {at_05['recall']:8.4f} {at_05['f1']:9.4f}")
print(f"{f'threshold {THRESHOLD:.2f}':16s} {at_t['precision']:8.4f} {at_t['recall']:8.4f} {at_t['f1']:9.4f}")
print(f"ROC-AUC {auc:.4f} | predicted in {predict_seconds:.0f}s")

layer_weights = {n: round(float(w), 4) for n, w in zip(LAYER_NAMES, model.layer_weights().detach())}
n_trained_best = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nbest epoch {best_epoch}: top {unfrozen_blocks(best_epoch)}/{N_BLOCKS} blocks trained "
      f"({n_trained_best:,} weights incl. the head)")
print("\nlayer mix after training:", layer_weights)
print(f"reference: best frozen-layer probe AUC {PROBE_BEST['auc']:.4f} ({PROBE_BEST_LAYER}) | "
      f"TF-IDF on the same text: AUC 0.708, macro-F1 0.653")

p, r, f, support = precision_recall_fscore_support(dev_labels, dev_preds, labels=[0, 1], zero_division=0)
print(f"\nper class      P      R      F1   gold")
for k in (0, 1):
    print(f"  label {k}  {p[k]:6.3f} {r[k]:6.3f} {f[k]:6.3f} {support[k]:6d}")

cm = confusion_matrix(dev_labels, dev_preds, labels=[0, 1])
print(f"\nconfusion matrix (rows = gold, columns = predicted)\n          pred 0  pred 1\n  gold 0  {cm[0, 0]:6d}  {cm[0, 1]:6d}\n  gold 1  {cm[1, 0]:6d}  {cm[1, 1]:6d}")


# ------------------------------------------------------------
# Errors by document length (characters), in 4 equal groups
# ------------------------------------------------------------

chars = np.array([len(c["text"]) for c in dev_cases])
edges = np.percentile(chars, [0, 25, 50, 75, 100])

print(f"\n{'length (chars)':>22s} {'cases':>6s} {'macro-F1':>9s} {'false +':>8s} {'false -':>8s}")

length_groups = []

for g in range(4):

    sel = (chars >= edges[g]) & ((chars <= edges[g + 1]) if g == 3 else (chars < edges[g + 1]))

    if sel.sum() == 0:
        continue

    s = binary_scores(dev_labels[sel], dev_preds[sel])
    fp = int(((dev_preds == 1) & (dev_labels == 0) & sel).sum())
    fn = int(((dev_preds == 0) & (dev_labels == 1) & sel).sum())

    length_groups.append({"min_chars": int(edges[g]), "max_chars": int(edges[g + 1]), "cases": int(sel.sum()),
                          "f1": s["f1"], "false_pos": fp, "false_neg": fn})

    print(f"{int(edges[g]):>10d} - {int(edges[g + 1]):>9d} {int(sel.sum()):6d} {s['f1']:9.4f} {fp:8d} {fn:8d}")


# ------------------------------------------------------------
# Which chunks matter?
#   1. average attention weight per chunk position (counted from the end
#      for DOC_PART "end", from the start otherwise)
#   2. macro-F1 when the classifier sees ONLY one chunk position
# ------------------------------------------------------------

from_end = DOC_PART == "end"

attention_by_position = {}
for w in dev_attention:
    for k, value in enumerate(w[::-1] if from_end else w):
        attention_by_position.setdefault(k, []).append(float(value))

single_chunk = {}

for k in range(max(len(v) for v in dev_vectors)):

    subset, idx = [], []

    for i, v in enumerate(dev_vectors):
        if len(v) > k:
            subset.append(v[len(v) - 1 - k:len(v) - k] if from_end else v[k:k + 1])
            idx.append(i)

    if not subset:
        continue

    probs_k, _ = classify_vectors(model, subset)
    single_chunk[k] = {"cases": len(idx), "f1": binary_scores(dev_labels[idx], (probs_k >= THRESHOLD).astype(int))["f1"]}

name = "chunk (0 = last)" if from_end else "chunk (0 = first)"

if DOC_PART == "start_end":
    print("(start_end: the first chunks come from the start of the document, the last ones from its end)")
print(f"\n{name:18s} {'mean attention':>15s} {'F1 with only this chunk':>24s}")

for k in sorted(attention_by_position):
    print(f"{k:18d} {np.mean(attention_by_position[k]):15.3f} {single_chunk.get(k, {}).get('f1', 0):24.4f}")

print(f"{'all chunks':18s} {'':15s} {at_t['f1']:24.4f}")


# ------------------------------------------------------------
# False positives / negatives: saved for reading
# ------------------------------------------------------------

errors = []

for case, label, pred, prob in zip(dev_cases, dev_labels, dev_preds, dev_probs):
    if label != pred:
        errors.append({"id": case["id"], "gold": int(label), "pred": int(pred), "prob_1": round(float(prob), 4),
                       "type": "false positive" if pred == 1 else "false negative",
                       "chars": len(case["text"]), "end_of_text": case["text"][-600:]})

fp_chars = [e["chars"] for e in errors if e["type"] == "false positive"]
fn_chars = [e["chars"] for e in errors if e["type"] == "false negative"]
ok_chars = [len(c["text"]) for c, l, p in zip(dev_cases, dev_labels, dev_preds) if l == p]

print(f"\nmedian length (chars): correct {np.median(ok_chars):.0f} | "
      f"false positives {np.median(fp_chars) if fp_chars else 0:.0f} ({len(fp_chars)}) | "
      f"false negatives {np.median(fn_chars) if fn_chars else 0:.0f} ({len(fn_chars)})")

with open(OUTPUT_DIR / "dev_errors.json", "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=1)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results = {
    "run": RUN_NAME,
    "model": "Architecture-1 decoder + layer freezing / gradual unfreezing, mix of all layers + token attention + chunk BiLSTM/attention classifier",
    "trained_on": "train only (dev used to pick the best epoch and the threshold)",
    "doc_part": DOC_PART, "doc_tokens": DOC_TOKENS, "chunk_len": CHUNK_LEN,
    "freezing": {"method": "gradual unfreezing + discriminative learning rates",
                 "unfreeze_fractions": UNFREEZE_FRACTIONS, "lr_top_block": LR_TOP_BLOCK,
                 "layer_lr_decay": LAYER_LR_DECAY, "blocks": N_BLOCKS,
                 "blocks_trained_at_best_epoch": unfrozen_blocks(best_epoch)},
    "parameters_total": n_total, "parameters_head": n_head,
    "parameters_trained_at_best_epoch": n_trained_best,
    "best_epoch": best_epoch, "train_minutes": train_minutes, "peak_gpu_gb": peak_gpu_gb,
    "dev_predict_seconds": predict_seconds,
    "threshold": THRESHOLD,
    "layer_mix": layer_weights,
    "probe_frozen_features": {"best_layer": PROBE_BEST_LAYER, "auc": PROBE_BEST["auc"], "f1": PROBE_BEST["f1"],
                              "by_layer": probe_results,
                              "cosine_raw": cos_raw, "cosine_standardized": cos_std},
    "signal_check_tfidf": signal_check,
    "dev_at_0.5": at_05, "dev_at_threshold": at_t, "dev_auc": auc,
    "dev_confusion_matrix": cm.tolist(),
    "dev_by_length": length_groups,
    "attention_by_chunk": {k: float(np.mean(v)) for k, v in attention_by_position.items()},
    "f1_single_chunk": single_chunk,
    "history": history,
}

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with open(OUTPUT_DIR / "dev_probabilities.json", "w", encoding="utf-8") as f:
    json.dump({c["id"]: round(float(p), 5) for c, p in zip(dev_cases, dev_probs)}, f)

print("\nSaved:", OUTPUT_DIR / "results.json", ", dev_errors.json, dev_probabilities.json")

DEV (p4_arch1_unfreeze, best epoch 3)
                  macro-P  macro-R  macro-F1
threshold 0.50     0.7397   0.5875    0.5097
threshold 0.22     0.6489   0.6479    0.6473
ROC-AUC 0.6785 | predicted in 69s

best epoch 3: top 6/12 blocks trained (46,319,889 weights incl. the head)


per class      P      R      F1   gold
  label 0   0.637  0.688  0.662    994
  label 1   0.661  0.608  0.633    994

confusion matrix (rows = gold, columns = predicted)
          pred 0  pred 1
  gold 0     684     310
  gold 1     390     604

        length (chars)  cases  macro-F1  false +  false -
       904 -     10667    496    0.6796       84       74
     10667 -     16700    498    0.6695       62      102
     16700 -     26562    496    0.6347       68      110
     26562 -    260198    498    0.5976       96      104

chunk (0 = last)    mean attention  F1 with only this chunk
                 0           0.400                   0.5805
                 1           0.296                   0.5441

In [16]:
# ============================================================
# CELL 16 — PREDICT TEST + WRITE prediction.json + submission.zip
#   format: [{"id": "case_001", "pred": 1}, ...]
# ============================================================

t0 = time.time()
test_probs, _ = classify_vectors(model, case_vectors(model, test_cases))
print(f"Predicted {len(test_cases)} test cases in {time.time() - t0:.0f}s (threshold {THRESHOLD:.2f})")

submission = [{"id": case["id"], "pred": int(prob >= THRESHOLD)} for case, prob in zip(test_cases, test_probs)]


# ------------------------------------------------------------
# Format checks
# ------------------------------------------------------------

assert len(submission) == len(test_cases)
assert [s["id"] for s in submission] == [c["id"] for c in test_cases]
assert all(isinstance(s["pred"], int) and s["pred"] in (0, 1) for s in submission)

counts = Counter(s["pred"] for s in submission)
print(f"✅ {len(submission)} predictions | pred 0: {counts[0]} | pred 1: {counts[1]}")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=1)

with open(OUTPUT_DIR / "test_probabilities.json", "w", encoding="utf-8") as f:
    json.dump({c["id"]: round(float(p), 5) for c, p in zip(test_cases, test_probs)}, f)

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("submission.zip:", zf.namelist())

print("Saved:", ZIP_FILE)

Predicted 1517 test cases in 53s (threshold 0.22)
✅ 1517 predictions | pred 0: 862 | pred 1: 655
submission.zip: ['prediction.json']
Saved: /kaggle/working/p4_arch1_unfreeze/submission.zip
